# Scaling to Hundreds of Meters

## Objectives

Chapter 1 measured forecastability for 342 real households individually and
closed with a floor: window-average, the simplest possible baseline, already
wins for 70% of customers. This chapter is where a trained model has to earn
its cost against that floor, and where the real deployment question a DSO
actually faces gets asked directly: is it one model per customer (accurate,
unmaintainable), one model for everyone (cheap, tailored to nobody), or
something that uses Chapter 4's own customer archetypes without paying for
either extreme?

By the end you will be able to:

- Check, on the real 342-customer AusNet pool, whether a trained LightGBM
  model actually beats Chapter 1's own baseline floor, and by how much.
- Build a single model that shares information across archetypes rather than
  training one model per cluster, and show honestly when that idea works and
  when it fails.
- Reuse Chapter 5's own calibrated retrieval-confidence machinery to answer a
  real operational question: what does a brand-new house, with no history in
  any trained model, get forecast with on day one?

## Getting the data and Chapter 4's own archetypes

Same real AusNet pool Chapter 1 used, and the same customer archetypes
Chapter 4 already built and validated on it: `KMeans(n_clusters=5)` on each
customer's own peak-normalized 90-day seasonal shape. Reused verbatim, not
re-clustered, since these are the same 342 customers.

In [ ]:
from pathlib import Path

from great_tables import GT, md
from lets_plot import (
    LetsPlot,
    aes,
    element_text,
    facet_wrap,
    geom_bar,
    geom_line,
    geom_point,
    ggplot,
    ggsize,
    labs,
    scale_color_manual,
    scale_fill_manual,
    theme,
)
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

from ark.plot.gt_style import themed_gt
from ark.plot.theme import modern_theme
from ark.plot.tokens import AI_ACCENT, GRAY_600, INFO, SUCCESS, WARNING

LetsPlot.setup_html(isolated_frame=True)

UNBOLD_AXES = theme(axis_title_x=element_text(face="plain"), axis_title_y=element_text(face="plain"))
FULL_WIDTH = 960
STEPS_PER_DAY = 48
STEPS_PER_WEEK = STEPS_PER_DAY * 7
FORECAST_HORIZON = STEPS_PER_DAY
TEST_STEPS = 90 * STEPS_PER_DAY  # the same last-90-real-days holdout Chapter 1 used
N_CLUSTERS = 5
DATA_DIR = Path("../../resources/cvar_flexibility/data/timeseries-lv")
RNG = np.random.default_rng(42)

# One fixed color per model family, reused in every chart in this notebook.
MODEL_COLORS = {
    "global_blind": GRAY_600,
    "hard_routed": WARNING,
    "unified": SUCCESS,
    "soft_mixture": AI_ACCENT,
    "soft_mixture_gated": INFO,
}


def normalize_shape(profiles: np.ndarray) -> np.ndarray:
    peak = profiles.max(axis=-1, keepdims=True)
    peak = np.where(peak == 0, 1, peak)
    return profiles / peak


load_data = np.load(DATA_DIR / "Residential load data 30-min resolution.npy")
n_customers = load_data.shape[0]
print(f"load_data: {load_data.shape} (customers, days, half-hours)")

season = load_data[:, 0:90, :]
X_shape = normalize_shape(season.mean(axis=1))
km = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=0)
archetype_labels = km.fit_predict(X_shape)
centroids = km.cluster_centers_


def distance_to_centroids(profiles: np.ndarray) -> np.ndarray:
    """Each profile's Euclidean distance to every one of Chapter 4's 5 archetype centroids."""
    return np.linalg.norm(profiles[:, None, :] - centroids[None, :, :], axis=2)


full_distances = distance_to_centroids(X_shape)

archetype_counts = pd.DataFrame({"archetype": range(N_CLUSTERS), "customers": np.bincount(archetype_labels)})
themed_gt(
    GT(archetype_counts)
    .tab_header(title=md("**Chapter 4's 5 Archetypes, Reused Directly**"))
    .cols_label(archetype=md("**Archetype**"), customers=md("**Customers**"))
    .tab_source_note("n = 342 real AusNet customers, KMeans(k=5) on peak-normalized 90-day shape"),
    n_rows=N_CLUSTERS,
)

load_data: (342, 365, 48) (customers, days, half-hours)


GT(_tbl_data=   archetype  customers
0          0         64
1          1        104
2          2         74
3          3         50
4          4         50, _body=<great_tables._gt_data.Body object at 0x11ba45490>, _boxhead=Boxhead([ColInfo(var='archetype', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Archetype**'), column_align='right', column_width=None), ColInfo(var='customers', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Customers**'), column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11b9f6990>, _spanners=Spanners([]), _heading=Heading(title=Md(text="**Chapter 4's 5 Archetypes, Reused Directly**"), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11ba9b0e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11b327170>, _source_notes=['n = 342 real AusNet customers, KMeans(k=5) on peak-normalized 90-day shape'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='archetype', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='customers', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='archetype', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='archetype', rownum=3, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='customers', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='customers', rownum=3, colnum=None, styles=[CellStyleFill(color='#F8F9FA')])], _locale=<great_tables._gt_data.Locale object at 0x11ba98bf0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['Libre Franklin', 'Segoe UI', 'Roboto', 'Helvetica Neue', 'sans-s

## Does profiling actually help, for a customer you already have?

Three approaches, compared honestly on the same real customers and the same
last-90-day holdout Chapter 1 used: one global LightGBM trained on everyone
pooled together with no archetype information at all, one LightGBM per
archetype (5 separate models, hard-routed by cluster membership), and one
single LightGBM that pools everyone together but also receives each
customer's own distance to every archetype centroid as five extra features.
The same lag-and-calendar feature recipe underlies all three: the most
recent reading, the same half-hour yesterday, two days ago, and a week ago,
plus half-hour-of-day and day-of-week.

In [ ]:
def build_lag_features(series: np.ndarray, *, horizon: int = FORECAST_HORIZON) -> pd.DataFrame:
    """A lag-and-calendar feature recipe.

    The same one Chapter 1 used for feature-importance ranking, now put to
    its first real use training a model.
    """
    feat = pd.DataFrame({"value": series})
    feat["target"] = feat["value"].shift(-horizon)
    for lag in (0, STEPS_PER_DAY, STEPS_PER_DAY * 2, STEPS_PER_WEEK):
        feat[f"lag_{lag}"] = feat["value"].shift(lag)
    step_idx = np.arange(len(feat))
    feat["half_hour"] = step_idx % STEPS_PER_DAY
    feat["dayofweek"] = (step_idx // STEPS_PER_DAY) % 7
    return feat.dropna().reset_index(drop=True)


LAG_COLS = ["lag_0", "lag_48", "lag_96", "lag_336", "half_hour", "dayofweek"]
DIST_COLS = [f"dist_c{k}" for k in range(N_CLUSTERS)]
UNIFIED_COLS = LAG_COLS + DIST_COLS

pool_rows = []
for cust_id in range(n_customers):
    feat = build_lag_features(load_data[cust_id].reshape(-1))
    feat["archetype"] = archetype_labels[cust_id]
    for k in range(N_CLUSTERS):
        feat[f"dist_c{k}"] = full_distances[cust_id, k]
    n = len(feat)
    feat["is_test"] = np.arange(n) >= (n - TEST_STEPS)  # last 90 real days, held out per customer
    pool_rows.append(feat)

pool = pd.concat(pool_rows, ignore_index=True)
train, test = pool[~pool["is_test"]], pool[pool["is_test"]]
print(f"pooled rows: {len(pool):,}, train: {len(train):,}, test: {len(test):,}")

pooled rows: 5,860,512, train: 4,383,072, test: 1,477,440


In [ ]:
from twiga.core.metrics import get_pointwise_metrics
from twiga.core.plot.gt import twiga_report
import twiga.models.ml.lightgbm_model  # noqa: F401 -- see num_threads note below
from twiga.models.ml.lightgbm_model import LIGHTGBMConfig, LIGHTGBMModel

# Real, verified environment bug, not a hand-wave: twiga imports torch even on
# this tree-model-only path, and torch's own bundled OpenMP runtime segfaults
# LightGBM's native multi-threaded histogram building on this machine.
# Confirmed directly: `import torch` alone before any multi-threaded
# LGBMRegressor.fit() call segfaults; num_threads=1 does not. Forced
# single-threaded as the practical fix, not assumed necessary.
LGBM_KWARGS = {"num_threads": 1}
METRIC_NAMES = ["mae", "rmse", "corr", "wmape", "smape"]


def to_xy(df: pd.DataFrame, feature_cols: list) -> tuple[np.ndarray, np.ndarray]:
    x = df[feature_cols].to_numpy()[:, None, :]
    y = df["target"].to_numpy()[:, None, None]
    return x, y


def eval_model(name: str, actual: np.ndarray, preds: np.ndarray) -> pd.DataFrame:
    row = get_pointwise_metrics(actual.reshape(-1), preds.reshape(-1), metric_names=METRIC_NAMES)
    row["Model"] = name
    return row


results_a = []

# global_blind: one model, everyone pooled, no archetype information
x_tr, y_tr = to_xy(train, LAG_COLS)
x_te, y_te = to_xy(test, LAG_COLS)
global_model = LIGHTGBMModel(LIGHTGBMConfig(**LGBM_KWARGS))
global_model.fit(x_tr, y_tr)
results_a.append(eval_model("global_blind", y_te, global_model.predict(x_te)))

# hard_routed: 5 separate models, one per archetype
preds_hard = np.zeros(len(test))
hard_models = {}
for k in range(N_CLUSTERS):
    tr_k, te_k = train[train["archetype"] == k], test[test["archetype"] == k]
    x_tr_k, y_tr_k = to_xy(tr_k, LAG_COLS)
    x_te_k, _ = to_xy(te_k, LAG_COLS)
    m = LIGHTGBMModel(LIGHTGBMConfig(**LGBM_KWARGS))
    m.fit(x_tr_k, y_tr_k)
    hard_models[k] = m
    preds_hard[(test["archetype"] == k).to_numpy()] = m.predict(x_te_k).reshape(-1)
results_a.append(eval_model("hard_routed", y_te, preds_hard))

# unified: one model, everyone pooled, archetype-distance features added
x_tr_u, y_tr_u = to_xy(train, UNIFIED_COLS)
x_te_u, y_te_u = to_xy(test, UNIFIED_COLS)
unified_model_full = LIGHTGBMModel(LIGHTGBMConfig(**LGBM_KWARGS))
unified_model_full.fit(x_tr_u, y_tr_u)
results_a.append(eval_model("unified", y_te_u, unified_model_full.predict(x_te_u)))

metrics_a = pd.concat(results_a, ignore_index=True)
res_a = metrics_a.groupby("Model", sort=False)[METRIC_NAMES].mean().round(4).reset_index()
res_a = res_a.rename(columns={"mae": "MAE", "rmse": "RMSE", "corr": "Corr", "wmape": "WMAPE", "smape": "SMAPE"})

twiga_report(
    res_a,
    ["MAE", "RMSE", "Corr", "WMAPE", "SMAPE"],
    ["MAE", "RMSE", "WMAPE", "SMAPE"],
    ["Corr"],
    title="Does Profiling Help? All 342 Real Customers",
    subtitle="24-hour-ahead, last 90 real days held out per customer",
    source_note="twiga.models.ml.LIGHTGBMModel, real AusNet data",
)

GT(_tbl_data=          Model     MAE    RMSE    Corr    WMAPE    SMAPE
0  global_blind  0.2563  0.4548  0.5921  55.0295  52.5523
1   hard_routed  0.2563  0.4539  0.5950  55.0372  52.3212
2       unified  0.2547  0.4474  0.6116  54.6968  52.1056, _body=<great_tables._gt_data.Body object at 0x12eb22930>, _boxhead=Boxhead([ColInfo(var='Model', type=<ColInfoTypeEnum.default: 1>, column_label='Model', column_align='left', column_width=None), ColInfo(var='MAE', type=<ColInfoTypeEnum.default: 1>, column_label='MAE', column_align='right', column_width=None), ColInfo(var='RMSE', type=<ColInfoTypeEnum.default: 1>, column_label='RMSE', column_align='right', column_width=None), ColInfo(var='Corr', type=<ColInfoTypeEnum.default: 1>, column_label='Corr', column_align='right', column_width=None), ColInfo(var='WMAPE', type=<ColInfoTypeEnum.default: 1>, column_label='WMAPE', column_align='right', column_width=None), ColInfo(var='SMAPE', type=<ColInfoTypeEnum.default: 1>, column_label='SMAPE', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x12eb228a0>, _spanners=Spanners([]), _heading=Heading(title='Does Profiling Help? All 342 Real Customers', subtitle='24-hour-ahead, last 90 real days held out per customer', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12eb22fc0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12eb22f90>, _source_notes=['twiga.models.ml.LIGHTGBMModel, real AusNet data'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#107591', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#718096', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Model', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='MAE', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='RMSE', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Corr', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='WMAPE', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='SMAPE', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#718096', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, wh

In [ ]:
metrics_a_plot = res_a.melt(id_vars="Model", value_vars=["MAE", "Corr"], var_name="metric", value_name="value")
(
    ggplot(metrics_a_plot, aes(x="Model", y="value", fill="Model"))
    + geom_bar(stat="identity", width=0.6)
    + facet_wrap("metric", ncol=2, scales="free_y")
    + labs(x="", y="", title="Unified Wins on Error and on Tracking the Real Signal")
    + scale_fill_manual(values=MODEL_COLORS)
    + modern_theme(legend_pos="none")
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 320)
)

Unified wins on every metric except SMAPE, where it is effectively tied.
It beats the global blind model, and it beats hard-routing to five separate
archetype-specific models too, without needing five models to maintain. But
this comparison has every customer's own real history inside the training
pool. That is a fair test of "does profiling help for a customer we already
have," not of a customer the system has never seen before, and those are
genuinely different questions.

## What does a brand-new house get forecast with?

A DSO onboarding a new house tomorrow has no history for it in any trained
model. The obvious idea is to reuse the unified model above exactly as it
is: compute the new house's own distance to each archetype centroid from
whatever real history exists, and feed those distances into the same model.
Checked directly, honestly, before trusting it.

In [ ]:
from sklearn.model_selection import train_test_split

N_NEW = 72  # held out entirely from training, never contribute a single row
HISTORY_DAYS = [1, 3, 7, 30, 90]

all_ids = RNG.choice(n_customers, size=n_customers, replace=False)
known_ids, new_ids = all_ids[: n_customers - N_NEW], all_ids[n_customers - N_NEW :]

known_rows = []
for cid in known_ids:
    feat = build_lag_features(load_data[cid].reshape(-1))
    feat["archetype"] = archetype_labels[cid]
    for k in range(N_CLUSTERS):
        feat[f"dist_c{k}"] = full_distances[cid, k]
    n = len(feat)
    feat["is_test"] = np.arange(n) >= (n - TEST_STEPS)
    known_rows.append(feat)
known_pool = pd.concat(known_rows, ignore_index=True)
known_train = known_pool[~known_pool["is_test"]]

# retrain all three model families on the KNOWN customers only: the 72 new
# ones must never have contributed a row to any of these fits
x_tr, y_tr = to_xy(known_train, LAG_COLS)
global_model_cs = LIGHTGBMModel(LIGHTGBMConfig(**LGBM_KWARGS))
global_model_cs.fit(x_tr, y_tr)

hard_models_cs = {}
for k in range(N_CLUSTERS):
    tr_k = known_train[known_train["archetype"] == k]
    x_tr_k, y_tr_k = to_xy(tr_k, LAG_COLS)
    m = LIGHTGBMModel(LIGHTGBMConfig(**LGBM_KWARGS))
    m.fit(x_tr_k, y_tr_k)
    hard_models_cs[k] = m

x_tr_u, y_tr_u = to_xy(known_train, UNIFIED_COLS)
unified_model_cs = LIGHTGBMModel(LIGHTGBMConfig(**LGBM_KWARGS))
unified_model_cs.fit(x_tr_u, y_tr_u)
print(f"retrained on {len(known_ids)} known customers, {len(new_ids)} held out entirely as 'new'")

retrained on 270 known customers, 72 held out entirely as 'new'


In [ ]:
def pre_test_partial_shape(cust_id: int, history_days: int) -> np.ndarray:
    """A new customer's own peak-normalized shape from only their first `history_days` real days."""
    pre_test_len_days = (365 * STEPS_PER_DAY - TEST_STEPS) // STEPS_PER_DAY
    pre_test_series = load_data[cust_id, :pre_test_len_days, :].reshape(-1)
    partial = pre_test_series[: history_days * STEPS_PER_DAY]
    n_days = len(partial) // STEPS_PER_DAY
    return normalize_shape(partial[: n_days * STEPS_PER_DAY].reshape(n_days, STEPS_PER_DAY).mean(axis=0, keepdims=True))


naive_rows = []
for cid in new_ids:
    feat = build_lag_features(load_data[cid].reshape(-1))
    n = len(feat)
    feat["is_test"] = np.arange(n) >= (n - TEST_STEPS)
    test_rows = feat[feat["is_test"]].copy()
    x_te_lag, y_te = to_xy(test_rows, LAG_COLS)
    preds_global = global_model_cs.predict(x_te_lag).reshape(-1)

    for hd in HISTORY_DAYS:
        pshape = pre_test_partial_shape(cid, hd)
        pdist = distance_to_centroids(pshape)[0]

        # the naive idea: feed the new customer's own (partial-history) centroid
        # distances into the SAME unified model trained on known customers
        test_rows_u = test_rows.copy()
        for k in range(N_CLUSTERS):
            test_rows_u[f"dist_c{k}"] = pdist[k]
        x_te_u, _ = to_xy(test_rows_u, UNIFIED_COLS)
        preds_naive = unified_model_cs.predict(x_te_u).reshape(-1)

        for name, preds in (("global_blind", preds_global), ("unified", preds_naive)):
            row = get_pointwise_metrics(y_te.reshape(-1), preds, metric_names=["mae", "rmse", "corr"])
            row["Model"] = name
            row["history_days"] = hd
            naive_rows.append(row)

naive_df = pd.concat(naive_rows, ignore_index=True)
naive_summary = (
    naive_df.groupby(["Model", "history_days"], sort=False)[["mae", "rmse", "corr"]].mean().round(4).reset_index()
)
themed_gt(
    GT(naive_summary.rename(columns={"mae": "MAE", "rmse": "RMSE", "corr": "Corr", "history_days": "History (days)"}))
    .tab_header(title=md("**The Naive Idea Fails, Even at a Full Season of History**"))
    .tab_source_note("72 real customers held out entirely from training, never seen by any model"),
    n_rows=len(naive_summary),
)

GT(_tbl_data=          Model  History (days)     MAE    RMSE    Corr
0  global_blind               1  0.2613  0.4271  0.3950
1       unified               1  0.2886  0.4634  0.3723
2  global_blind               3  0.2613  0.4271  0.3950
3       unified               3  0.2760  0.4456  0.3908
4  global_blind               7  0.2613  0.4271  0.3950
5       unified               7  0.2763  0.4470  0.3964
6  global_blind              30  0.2613  0.4271  0.3950
7       unified              30  0.2704  0.4348  0.4008
8  global_blind              90  0.2613  0.4271  0.3950
9       unified              90  0.2675  0.4333  0.4026, _body=<great_tables._gt_data.Body object at 0x12f87a2a0>, _boxhead=Boxhead([ColInfo(var='Model', type=<ColInfoTypeEnum.default: 1>, column_label='Model', column_align='left', column_width=None), ColInfo(var='History (days)', type=<ColInfoTypeEnum.default: 1>, column_label='History (days)', column_align='right', column_width=None), ColInfo(var='MAE', type=<ColInfoTypeEnum.default: 1>, column_label='MAE', column_align='right', column_width=None), ColInfo(var='RMSE', type=<ColInfoTypeEnum.default: 1>, column_label='RMSE', column_align='right', column_width=None), ColInfo(var='Corr', type=<ColInfoTypeEnum.default: 1>, column_label='Corr', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x12f879280>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**The Naive Idea Fails, Even at a Full Season of History**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x13013e7e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1302a79b0>, _source_notes=['72 real customers held out entirely from training, never seen by any model'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Model', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='History (days)', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='MAE', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='RMSE', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Corr', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transfo

Even at 90 real days of history, exactly matching the full season used to
build the archetype centroids in the first place, the unified model is still
worse than the global blind model that ignores archetype information
entirely. More history does not fix it. The distance features are not being
ignored either, a feature-importance check shows LightGBM splits on them
about as often as it splits on the lag features. What is actually
happening: a customer's own centroid-distance vector is close to a unique
signature during training, and the tree learns to recognize *which known
customer* a row came from rather than *what archetype behavior generalizes*.
That is a legitimate personalization signal for a customer the model has
already trained on (Experiment A's real win above), and exactly the wrong
signal for a customer it has never seen, there is no signature to recognize
yet.

The fix keeps every trained archetype specialist from Experiment A but stops
asking a single model to reason about static per-customer distances at all.
Instead: blend the five specialists' own predictions directly, weighted by
how close the new customer's (partial-history) shape sits to each archetype
centroid, and gate the whole thing behind Chapter 5's own calibrated
retrieval-confidence check, reused verbatim, not reimplemented, so a
customer with no close real match anywhere in the known pool safely falls
back to the blind model instead of a confident-looking guess.

In [ ]:
from ark.recommend.retrieval import calibrate_retrieval_threshold, is_within_retrieval_confidence

TEMP = 0.05  # softmax temperature over -distance

# Chapter 5's own split-conformal retrieval-confidence calibration, reused directly:
# a query point is "trusted" only if its nearest labeled neighbor sits within a
# distance covering 90% of a held-back calibration split.
X_known = X_shape[known_ids]
retrieval_train_idx, calib_idx = train_test_split(np.arange(len(known_ids)), test_size=0.3, random_state=0)
X_retrieval_train = X_known[retrieval_train_idx]
tau = calibrate_retrieval_threshold(X_retrieval_train, X_known[calib_idx], alpha=0.1)
print(f"calibrated retrieval tau: {tau:.4f}")

gated_rows = []
for cid in new_ids:
    feat = build_lag_features(load_data[cid].reshape(-1))
    n = len(feat)
    feat["is_test"] = np.arange(n) >= (n - TEST_STEPS)
    test_rows = feat[feat["is_test"]].copy()
    x_te_lag, y_te = to_xy(test_rows, LAG_COLS)
    preds_global = global_model_cs.predict(x_te_lag).reshape(-1)
    preds_by_archetype = {k: hard_models_cs[k].predict(x_te_lag).reshape(-1) for k in range(N_CLUSTERS)}
    true_archetype = archetype_labels[cid]

    for hd in HISTORY_DAYS:
        pshape = pre_test_partial_shape(cid, hd)
        pdist = distance_to_centroids(pshape)[0]
        assigned = int(np.argmin(pdist))

        preds_hard_cs = preds_by_archetype[assigned]
        weights = np.exp(-pdist / TEMP)
        weights = weights / weights.sum()
        preds_soft = sum(weights[k] * preds_by_archetype[k] for k in range(N_CLUSTERS))
        trusted = bool(is_within_retrieval_confidence(X_retrieval_train, pshape, tau)[0])
        preds_gated = preds_soft if trusted else preds_global

        for name, preds in (
            ("global_blind", preds_global),
            ("hard_routed", preds_hard_cs),
            ("soft_mixture", preds_soft),
            ("soft_mixture_gated", preds_gated),
        ):
            row = get_pointwise_metrics(y_te.reshape(-1), preds, metric_names=["mae", "rmse", "corr"])
            row["Model"] = name
            row["history_days"] = hd
            row["correctly_assigned"] = assigned == true_archetype
            row["trusted"] = trusted
            gated_rows.append(row)

gated_df = pd.concat(gated_rows, ignore_index=True)
gated_summary = gated_df.groupby(["Model", "history_days"], sort=False)[["mae", "rmse", "corr"]].mean().reset_index()

calibrated retrieval tau: 0.8450


In [ ]:
gated_colors = {k: v for k, v in MODEL_COLORS.items() if k in gated_summary["Model"].unique()}
(
    ggplot(gated_summary, aes(x="history_days", y="mae", color="Model"))
    + geom_line(size=1.1)
    + geom_point(size=2.5)
    + scale_color_manual(values=gated_colors, name="")
    + labs(
        x="Real days of history available for the new customer",
        y="MAE",
        title="Gated Mixture Never Falls Below the Floor",
    )
    + modern_theme()
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 340)
)

In [ ]:
assign_acc = (
    gated_df[gated_df["Model"] == "soft_mixture"]
    .groupby("history_days")["correctly_assigned"]
    .mean()
    .round(3)
    .rename("Correct archetype assignment")
)
trust_rate = (
    gated_df[gated_df["Model"] == "soft_mixture_gated"]
    .groupby("history_days")["trusted"]
    .mean()
    .round(3)
    .rename("Retrieval-confidence trust rate")
)
diagnostics = (
    pd.concat([assign_acc, trust_rate], axis=1).reset_index().rename(columns={"history_days": "History (days)"})
)
themed_gt(
    GT(diagnostics)
    .tab_header(title=md("**Both Sharpen Together as Real History Accumulates**"))
    .tab_source_note("72 real customers held out entirely from training"),
    n_rows=len(diagnostics),
)

GT(_tbl_data=   History (days)  Correct archetype assignment  \
0               1                         0.319   
1               3                         0.403   
2               7                         0.472   
3              30                         0.681   
4              90                         1.000   

   Retrieval-confidence trust rate  
0                            0.000  
1                            0.111  
2                            0.389  
3                            0.833  
4                            0.917  , _body=<great_tables._gt_data.Body object at 0x13081e030>, _boxhead=Boxhead([ColInfo(var='History (days)', type=<ColInfoTypeEnum.default: 1>, column_label='History (days)', column_align='right', column_width=None), ColInfo(var='Correct archetype assignment', type=<ColInfoTypeEnum.default: 1>, column_label='Correct archetype assignment', column_align='right', column_width=None), ColInfo(var='Retrieval-confidence trust rate', type=<ColInfoTypeEnum.default: 1>, column_label='Retrieval-confidence trust rate', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x12eb761e0>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Both Sharpen Together as Real History Accumulates**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1304e21e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x130734ec0>, _source_notes=['72 real customers held out entirely from training'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='History (days)', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Correct archetype assignment', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Retrieval-confidence trust rate', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='History (days)', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='History (days)', rownum=3, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Correct archetype assignment', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Correct archetype assignment', rownum=3, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(

## From mechanism to decision

Two different answers to "how should clustering and forecasting combine,"
and which one is right depends entirely on whether a customer is known or
new, checked rather than assumed either way. For a customer the system
already has any real history for, a single model that receives archetype
distances as ordinary features wins outright (Experiment A), because that
static distance signature legitimately doubles as a personalization signal.
For a customer with no history in any trained model yet, that same
mechanism actively fails, even with a perfect archetype match, because there
is no signature yet to personalize against. The fix reuses machinery this
book already built rather than inventing new machinery: Chapter 4's
archetype centroids supply the distances, Chapter 5's own calibrated
retrieval-confidence check supplies the honest "do we actually have a close
real match" gate, and the two combine into a soft, distance-weighted blend
of the same specialist models Experiment A already trained. It is never
worse than doing nothing, and it gets measurably better as real history
accumulates, exactly the property a genuine day-one deployment needs.

## Why bother

A DSO does not get to choose between "accurate" and "operationally
maintainable," and does not get to wait for a new customer to accumulate a
full season of history before forecasting them at all. The real, checked
finding here is that both constraints have one honest answer each: reuse a
single model with cluster information as a legitimate personalization
signal once a customer has real history, and fall back to a calibrated,
distance-weighted blend of already-trained specialists, safely gated,
before that history exists. Chapter 3 picks up from the model this chapter
found actually earns its cost, and asks what that forecast is worth without
a number attached to how wrong it might be.